In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

In [5]:
# 1. Configuration & Bounding Boxes
# ==========================================
NUM_RECORDS = 10000
START_DATE = datetime(2025, 10, 1) # Simulating Q4 to capture winter weather
END_DATE = datetime(2026, 3, 31)

In [8]:
# Rough bounding boxes for target Budapest districts (V, VI, VII, IX, XI)
# Format: [Min_Lat, Max_Lat, Min_Lon, Max_Lon]
DISTRICT_BOUNDS = {
    'V_VI_VII': [47.4930, 47.5150, 19.0450, 19.0750], # Pest Center (Dense, short trips)
    'IX': [47.4650, 47.4900, 19.0600, 19.0950],       # Ferencváros
    'XI': [47.4500, 47.4850, 19.0150, 19.0550]        # Újbuda (Hilly, across the river)
}

In [9]:
# 2. Helper Functions
# ==========================================
def random_dates(start, end, n):
    """Generates random timestamps between two dates."""
    start_u = start.timestamp()
    end_u = end.timestamp()
    return pd.to_datetime(np.random.uniform(start_u, end_u, n), unit='s')

def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculates distance in km between two GPS coordinates."""
    R = 6371.0 # Earth radius in kilometers
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

In [10]:
# 3. Data Generation Pipeline
# ==========================================
print("Initializing delivery simulation for Budapest...")
np.random.seed(42) # For reproducibility

# 3.1 Generate Timestamps
timestamps = random_dates(START_DATE, END_DATE, NUM_RECORDS)

Initializing delivery simulation for Budapest...


In [11]:
# 3.2 Generate Locations (Pickups and Dropoffs)
# Randomly assign a district region for pickup and dropoff
pickup_regions = np.random.choice(list(DISTRICT_BOUNDS.keys()), NUM_RECORDS, p=[0.5, 0.25, 0.25])
dropoff_regions = np.random.choice(list(DISTRICT_BOUNDS.keys()), NUM_RECORDS, p=[0.4, 0.3, 0.3])

pickup_lats, pickup_lons, dropoff_lats, dropoff_lons = [], [], [], []

for p_reg, d_reg in zip(pickup_regions, dropoff_regions):
    p_bounds = DISTRICT_BOUNDS[p_reg]
    d_bounds = DISTRICT_BOUNDS[d_reg]

    pickup_lats.append(np.random.uniform(p_bounds[0], p_bounds[1]))
    pickup_lons.append(np.random.uniform(p_bounds[2], p_bounds[3]))

    dropoff_lats.append(np.random.uniform(d_bounds[0], d_bounds[1]))
    dropoff_lons.append(np.random.uniform(d_bounds[2], d_bounds[3]))

In [12]:
# 3.3 Weather & Courier Setup
weather_conditions = np.random.choice(
    ['Clear', 'Cloudy', 'Rain', 'Snow'],
    NUM_RECORDS,
    p=[0.40, 0.35, 0.20, 0.05]
)

courier_vehicles = np.random.choice(['Bicycle', 'E-Bike', 'Car', 'Scooter'], NUM_RECORDS)

In [13]:
# Create Base DataFrame
df = pd.DataFrame({
    'Order_ID': [f"ORD-{10000 + i}" for i in range(NUM_RECORDS)],
    'Order_Timestamp': timestamps,
    'Pickup_Lat': pickup_lats,
    'Pickup_Lon': pickup_lons,
    'Dropoff_Lat': dropoff_lats,
    'Dropoff_Lon': dropoff_lons,
    'Weather_Condition': weather_conditions,
    'Courier_Vehicle': courier_vehicles
})

In [15]:
# 4. Feature Engineering & Financial Logic
# ==========================================
print("Applying distance and financial logic...")

# Calculate straight-line distance, then multiply by 1.3 to simulate road routing
df['Est_Distance_km'] = haversine_distance(
    df['Pickup_Lat'], df['Pickup_Lon'], df['Dropoff_Lat'], df['Dropoff_Lon']
) * 1.3

Applying distance and financial logic...


In [16]:
# Base Payout: 600 HUF drop fee + 150 HUF per km
df['Base_Payout_HUF'] = 600 + (df['Est_Distance_km'] * 150)

In [17]:
# Peak Hour Multiplier (Dinner rush 18:00 - 21:00)
df['Is_Peak_Hour'] = df['Order_Timestamp'].dt.hour.isin([18, 19, 20])
peak_multiplier = np.where(df['Is_Peak_Hour'], 1.25, 1.0)

In [18]:
# Weather Multiplier (Platforms pay more when it rains/snows)
weather_mult_map = {'Clear': 1.0, 'Cloudy': 1.0, 'Rain': 1.4, 'Snow': 1.75}
weather_multiplier = df['Weather_Condition'].map(weather_mult_map)

In [19]:
# Final Payout Calculation (Rounded to nearest 10 HUF)
df['Final_Payout_HUF'] = np.round((df['Base_Payout_HUF'] * peak_multiplier * weather_multiplier) / 10) * 10

In [20]:
# Simulate Delivery Times (in minutes)
# Cars/Scooters are faster over long distances, Bikes faster in dense traffic
base_time = df['Est_Distance_km'] * np.where(df['Courier_Vehicle'].isin(['Car', 'Scooter']), 4, 6)
weather_delay = df['Weather_Condition'].map({'Clear': 0, 'Cloudy': 2, 'Rain': 10, 'Snow': 20})
random_variance = np.random.normal(loc=5, scale=3, size=NUM_RECORDS) # 5 min avg wait at restaurant

df['Actual_Delivery_Time_Min'] = np.round(base_time + weather_delay + random_variance).clip(lower=10)

In [21]:
# 5. Export Data
# ==========================================
# Sort chronologically to simulate a real database dump
df = df.sort_values('Order_Timestamp').reset_index(drop=True)

output_filename = "budapest_delivery_simulation_2026.csv"
df.to_csv(output_filename, index=False)
print(f"Success! {NUM_RECORDS} records generated and saved to {output_filename}.")
print("\nSample Data:")
print(df[['Order_Timestamp', 'Est_Distance_km', 'Weather_Condition', 'Final_Payout_HUF', 'Actual_Delivery_Time_Min']].head()) #first 5 rows

Success! 10000 records generated and saved to budapest_delivery_simulation_2026.csv.

Sample Data:
                Order_Timestamp  Est_Distance_km Weather_Condition  \
0 2025-10-01 00:03:01.948958397         7.936243             Clear   
1 2025-10-01 00:08:00.393591642         4.085517              Rain   
2 2025-10-01 00:13:46.128698111         1.212816            Cloudy   
3 2025-10-01 00:35:06.383081436         5.102808            Cloudy   
4 2025-10-01 00:59:10.514472485         6.198314             Clear   

   Final_Payout_HUF  Actual_Delivery_Time_Min  
0            1790.0                      49.0  
1            1700.0                      42.0  
2             780.0                      13.0  
3            1370.0                      28.0  
4            1530.0                      47.0  
